In [1]:
!pip install openai


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from openai import OpenAI

# Prototype only: configuration must come from the environment.
PROXY_URL = os.getenv("JLT_MODEL_BASE_URL")
VIRTUAL_KEY = os.getenv("JLT_MODEL_API_KEY")
MODEL = os.getenv("JLT_PRIMARY_MODEL")

if not all((PROXY_URL, VIRTUAL_KEY, MODEL)):
    raise RuntimeError("Model configuration is missing; no provider request was made.")

client = OpenAI(
    api_key=VIRTUAL_KEY,
    base_url=PROXY_URL,
)

print("Sending a request through the configured model proxy...\n")

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "あなたは親切なAIアシスタントです。"},
            {"role": "user", "content": "Azure Container Apps の長所を1文で教えてください。"},
        ],
        max_completion_tokens=1024,
    )
    print("【AIからの応答】")
    print(response.choices[0].message.content)
except Exception:
    print("The provider request failed. No credential or response detail was logged.")

Sending a request through the configured model proxy...

【AIからの応答】
Azure Container Apps の長所は、Kubernetes のような複雑な管理を必要とせず、サーバーレス環境でコンテナアプリケーションを簡単にデプロイ・スケーリングできる点です。


In [8]:
# This prototype cell reuses the environment-configured client from the prior cell.

# 社内データ（ダミー）の準備
# 実際のシステムでは、ここがファイル検索やデータベース検索になります。
dummy_documents = {
    "リモートワーク": "【リモートワーク規程】週3日までの在宅勤務を許可します。事前に上長の承認が必要です。コアタイムは10:00〜15:00です。",
    "交通費": "【交通費精算ルール】毎月末日までに、専用システム「T-Expense」から申請してください。月額上限は5万円までとなります。",
    "有給休暇": "【有給休暇について】入社半年後に10日付与されます。半休（午前休・午後休）の取得も可能です。",
}


def simple_search(query):
    for keyword, text in dummy_documents.items():
        if keyword in query:
            return text
    return "関連する社内規程が見つかりませんでした。"


user_question = "交通費の申請方法と期限を教えてください。"
print(f"ユーザーの質問: {user_question}\n")
retrieved_info = simple_search(user_question)
print(f"【システムログ】検索された情報:\n{retrieved_info}\n")

system_prompt = f"""
あなたは社内ヘルプデスクのAIアシスタントです。
以下の【社内ルール】のみに基づいて、ユーザーの質問に答えてください。
ルールに記載がないことは「わかりません」と答えてください。

【社内ルール】
{retrieved_info}
"""

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    )
    print("【AIからの回答】")
    print(response.choices[0].message.content)
except Exception:
    print("The provider request failed. No credential or response detail was logged.")

ユーザーの質問: 交通費の申請方法と期限を教えてください。

【システムログ】検索された情報:
【交通費精算ルール】毎月末日までに、専用システム「T-Expense」から申請してください。月額上限は5万円までとなります。

【AIからの回答】
交通費の申請方法と期限は以下の通りです。

1. **申請方法**：専用システム「T-Expense」から申請してください。
2. **申請期限**：毎月末日までに申請を完了してください。

このルールに従って、交通費の精算を行ってください。


In [5]:
import json
# ==========================================
# 2. 入力データ（処理したい長文テキスト）
# ==========================================
daily_report = """
本日、株式会社〇〇商事様を訪問し、新システムの提案を行いました。
機能面については「業務効率が上がりそうだ」と非常に前向きな反応をいただきました。
ただ、導入費用についての懸念があり、一旦持ち帰り検討となっています。
競合のA社も提案に入っているとのことなので、今週金曜日までに割引を含めた再見積もりを提出する必要があります。
至急、営業部長の承認をお願いします。
"""

print("【処理前の日報】")
print(daily_report)
print("-" * 40)

# ==========================================
# 3. プロンプト（抽出ルールの定義）
# ==========================================
# ポイント：「JSON形式で出力すること」「キーの名前と型」を厳密に指示する
system_prompt = """
あなたは優秀なデータ処理アシスタントです。
入力された営業日報から以下の情報を抽出し、指定されたJSONフォーマットで出力してください。
マークダウンの装飾（```jsonなど）や、JSON以外の説明文は一切含めないでください。

【抽出項目（JSONキー）】
- company_name: 顧客の企業名（文字列）
- reaction: 顧客の反応（"ポジティブ", "ネガティブ", "ニュートラル" のいずれか）
- issue: 懸念点や課題（文字列。なければ null）
- next_action: 次に行うべきアクションと期限（文字列）
- urgency: 緊急度（"高", "中", "低" のいずれか。至急の対応が必要な場合は"高"とする）
"""

# ==========================================
# 4. APIリクエストとデータ処理
# ==========================================
print("AIが日報を解析中...\n")

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": daily_report}
        ],
    )
    
    # AIからのテキスト応答を取得
    ai_response_text = response.choices[0].message.content.strip()
    
    # 取得した文字列をPythonの辞書（JSON）オブジェクトに変換
    ai_response_text = ai_response_text.removeprefix("```json\n").removesuffix("\n```")
    extracted_data = json.loads(ai_response_text)
    
    print("【抽出成功！処理後のデータ】")
    for key, value in extracted_data.items():
        print(f"{key}: {value}")

except json.JSONDecodeError:
    print("エラー：AIが正しいJSON形式で出力しませんでした。")
    print("AIの出力内容:", ai_response_text)
except Exception as e:
    print(f"エラーが発生しました: {e}")

【処理前の日報】

本日、株式会社〇〇商事様を訪問し、新システムの提案を行いました。
機能面については「業務効率が上がりそうだ」と非常に前向きな反応をいただきました。
ただ、導入費用についての懸念があり、一旦持ち帰り検討となっています。
競合のA社も提案に入っているとのことなので、今週金曜日までに割引を含めた再見積もりを提出する必要があります。
至急、営業部長の承認をお願いします。

----------------------------------------
AIが日報を解析中...

【抽出成功！処理後のデータ】
company_name: 株式会社〇〇商事
reaction: ポジティブ
issue: 導入費用についての懸念
next_action: 割引を含めた再見積もりを提出（今週金曜日まで）
urgency: 高


In [7]:
import time
# ==========================================
# 2. 比較したいモデルのリストを指定
# ==========================================
MODELS_TO_COMPARE = [
    "tsuzumi2",
    "gpt-5.6-luna",
    "gpt-5-nano" 
]

# ==========================================
# 3. 検証用プロンプトの準備
# ==========================================
# 例：日本語の細かいニュアンスや業務表現に関する質問
test_prompt = "敬語の「お越しになられる」という表現は正しいですか？理由と適切な表現を簡潔に教えてください。"

print(f"【検証プロンプト】{test_prompt}")
print("=" * 60)

# ==========================================
# 4. ループ処理による比較実行
# ==========================================
results = {}

for model_name in MODELS_TO_COMPARE:
    print(f"▶ [{model_name}] にリクエストを送信中...")
    
    start_time = time.time() # 処理時間の計測開始
    
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "あなたは正確で丁寧な日本語のアシスタントです。"},
                {"role": "user", "content": test_prompt}
            ],
        )
        
        elapsed_time = time.time() - start_time # 処理時間の計算
        content = response.choices[0].message.content
        
        # 結果の保存
        results[model_name] = {
            "time": elapsed_time,
            "content": content
        }
        
    except Exception as e:
        print(f"[{model_name}] でエラーが発生しました: {e}")

# ==========================================
# 5. 比較結果の表示
# ==========================================
print("=" * 60)
print("【比較検証結果】")
print("=" * 60)

for model_name, res in results.items():
    print(f"■ モデル名: {model_name}")
    print(f"⏱ 応答時間: {res['time']:.2f} 秒")
    print(f"💬 回答内容:{res['content']}")
    print("-" * 40)

【検証プロンプト】敬語の「お越しになられる」という表現は正しいですか？理由と適切な表現を簡潔に教えてください。
▶ [tsuzumi2] にリクエストを送信中...
▶ [gpt-5.6-luna] にリクエストを送信中...
▶ [gpt-5-nano] にリクエストを送信中...
【比較検証結果】
■ モデル名: tsuzumi2
⏱ 応答時間: 11.31 秒
💬 回答内容:**「お越しになられる」は文法的に誤りです。**

**理由**
1. **二重敬語**
   - 「お越しになる」だけでも「来る」の尊敬語です。
   - さらに「られる」を付けると、尊敬のレベルが二重になり、過剰な敬語（二重敬語）になります。

2. **「られる」の使い方**
   - 「られる」は受身・可能・自発などの意味を表す助動詞で、尊敬語の「なる」に直接付けることはできません。

**適切な表現**
- **「お越しになる」**（最も一般的）
- **「お越しくださる」**（「くださる」は「くれる」の尊敬語で、やや柔らかい印象）
- **「お越しいただく」**（「いただく」は「もらう」の謙譲語ですが、相手の行為を謙譲的に表すときに使う）

したがって、相手が来訪することを敬意を込めて言う場合は **「お越しになる」** が正解です。
----------------------------------------
■ モデル名: gpt-5.6-luna
⏱ 応答時間: 5.27 秒
💬 回答内容:「お越しになられる」は、原則として不適切です。  
「お越しになる」自体が「来る」の尊敬語で、さらに「〜られる」を重ねた**二重敬語**に当たるためです。

適切な表現は次のとおりです。

- 来ることを敬って言う：**お越しになる**
- より一般的な尊敬語：**いらっしゃる**
- 来訪を改まって言う：**お見えになる**

例：  
- × 先生がお越しになられました。  
- ○ 先生がお越しになりました。  
- ○ 先生がいらっしゃいました。
----------------------------------------
■ モデル名: gpt-5-nano
⏱ 応答時間: 26.22 秒
💬 回答内容:結論
「お越しになられる」は二重敬語とされ、現代の標準的な敬語と